# YOLO26 ThermalUAV2UAV baseline (Colab)

1. Set `REPO_URL` to your GitHub repo.
2. Runtime → Change runtime type → GPU.
3. Run all cells.
4. Download `reports/*.md` and `reports/*.json` (optionally sync `runs/` to Drive).

In [ ]:
REPO_URL = "https://github.com/YOUR_USER/Ther-Inf-multiclass-detection.git"
BRANCH = "main"
EPOCHS = 20
IMGSZ = 640
BATCH = 16
RUN_NAME = "uav2uav_yolo26n_baseline"
SAVE_RUNS_TO_DRIVE = True
DRIVE_ROOT = "/content/drive/MyDrive/ther_inf_runs"

In [ ]:
from google.colab import drive
import os

if SAVE_RUNS_TO_DRIVE:
    drive.mount("/content/drive")
    os.makedirs(DRIVE_ROOT, exist_ok=True)

In [ ]:
import os
import pathlib

os.chdir("/content")
!rm -rf repo
!git clone --branch {BRANCH} {REPO_URL} repo

cands = list(pathlib.Path("/content/repo").glob("**/train.py"))
assert cands, "train.py not found after clone"
ROOT = str(cands[0].parent)
print("ROOT=", ROOT)
os.chdir(ROOT)

In [ ]:
!pip -q install -r requirements.txt
!mkdir -p data
!git clone --depth 1 https://github.com/GabryV00/ThermalUAV2UAV_Dataset.git data/ThermalUAV2UAV_Dataset

In [ ]:
!python train.py --epochs {EPOCHS} --imgsz {IMGSZ} --batch {BATCH} --name {RUN_NAME}

In [ ]:
from pathlib import Path
import shutil

report_md = Path("reports") / f"{RUN_NAME}.md"
report_json = Path("reports") / f"{RUN_NAME}.json"
print("Report:", report_md if report_md.exists() else "(missing)")

if SAVE_RUNS_TO_DRIVE:
    dest = Path(DRIVE_ROOT) / RUN_NAME
    if dest.exists():
        shutil.rmtree(dest)
    shutil.copytree(Path("runs") / RUN_NAME, dest)
    if report_md.exists():
        shutil.copy2(report_md, Path(DRIVE_ROOT) / report_md.name)
    if report_json.exists():
        shutil.copy2(report_json, Path(DRIVE_ROOT) / report_json.name)
    print("Copied run + reports to Drive:", dest)

### After Colab
1. Download `reports/<run>.md` and `reports/<run>.json`.
2. Commit them to the repo.
3. Keep heavy `runs/` and `*.pt` files on Drive (they are gitignored).